# DQ: первичная проверка `commission_monthly` из `n_amt`

Тетрадка проверяет качество первичной загрузки таблицы `ods_alpha.scd1_mrc_pos_rent`, где месячная комиссия хранится в `n_amt`.

Что делает:
- техническая валидация загрузки (структура, свежесть, служебные поля);
- профилирование качества `n_amt` (null/zero/negative, распределение, выбросы);
- анализ дублей и конфликтов по ключу `c_nmrc + d_rent`;
- помесячная агрегация комиссии после дедупликации;
- маппинг `c_nmrc -> inn + agr_id` через `agr_terms/agreements/companies` и контроль покрытия;
- сверка с Excel (Jan-Apr) по ключу `inn+agr_id`;
- выбор одного общего мерчанта (авто) и детальная сверка по месяцам.

> При необходимости поменяйте пути к Excel и параметры подключения в конфиге ниже.

In [ ]:
import re
import time
from decimal import Decimal, InvalidOperation
from pathlib import Path

import numpy as np
import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))


def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def pick_col_robust(columns, candidates):
    cols = list(columns)
    norm = lambda s: re.sub(r'\s+', ' ', str(s).replace('\xa0', ' ').strip().lower())
    norm_map = {norm(c): c for c in cols}
    for c in candidates:
        if c in cols:
            return c
        nc = norm(c)
        if nc in norm_map:
            return norm_map[nc]
    return None


def to_num_series(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace('\xa0', '', regex=False)
         .str.replace(' ', '', regex=False)
         .str.replace(',', '.', regex=False),
        errors='coerce'
    )


def safe_divergence_pct(delta, reference):
    if pd.isna(reference) or reference == 0:
        return np.nan
    return abs(delta) / abs(reference) * 100.0

In [ ]:
# === Основные таблицы ===
mrc_table = 'ods_alpha.scd1_mrc_pos_rent'
agr_terms_table = 'ods_alpha.scd1_agr_terms'
agreements_table = 'ods_alpha.scd1_agreements'
companies_table = 'ods_alpha.scd1_companies'

# === Период сравнения ===
period_start = '2026-01-01'
period_end = '2026-04-30'
period_months = pd.date_range(period_start, period_end, freq='MS')

# === Excel-референсы ===
excel_header_default = 0
excel_header_by_month = {
    '2026-01': 1,
}
excel_reference_by_month = {
    '2026-01': '/home/jovyan/documents/Equaring/Data/01_Январь_2026.xlsx',
    '2026-02': '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx',
    '2026-03': '/home/jovyan/documents/Equaring/Data/03_Март_2026.xlsx',
    '2026-04': '/home/jovyan/documents/Equaring/Data/04_Апрель_2026.xlsx',
}

# Если хотите фиксированный кейс вместо авто-выбора общего мерчанта, заполните оба поля.
manual_inn_key = None
manual_agr_id_key = None

# === Подключение ===
impala_db = 'sandbox_ai'
impala_mem_limit = '8g'
impala_user_name = 'Shestopalov-VYur'

# === Выгрузка результатов ===
output_dir = Path('/home/jovyan/documents/Equaring/Data')
output_report_path = output_dir / 'commission_monthly_n_amt_dq_report.xlsx'
output_key_compare_csv = output_dir / 'commission_monthly_selected_key_compare.csv'

print('Период:', [d.strftime('%Y-%m') for d in period_months])
print('Таблица для проверки:', mrc_table)
print('Excel-референсы:')
for m, p in excel_reference_by_month.items():
    h = excel_header_by_month.get(m, excel_header_default)
    print(f'  {m}: {p} (header={h})')

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': impala_db},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': impala_user_name}
)
imp._init_connection()


def run_sql(sql_text, step_name='query', mem_limit=impala_mem_limit):
    start_ts = time.perf_counter()
    print(f'[{step_name}] start')
    with imp:
        imp.execute(f"set MEM_LIMIT={mem_limit}")
        df = imp.fetch(sql_text)
    elapsed = round(time.perf_counter() - start_ts, 2)
    rows = len(df) if isinstance(df, pd.DataFrame) else 0
    print(f'[{step_name}] done in {elapsed}s, rows={rows:,}')
    return df


def sql_escape(value):
    return str(value).replace("'", "''")


print('Impala connection initialized')

## 1) Техническая валидность таблицы

In [ ]:
schema_name, table_name = mrc_table.split('.', 1)

mrc_candidates_df = run_sql(
    f"show tables in {schema_name} like '*mrc*rent*'",
    step_name='show_mrc_candidates'
)
display(mrc_candidates_df)

mrc_describe_df = run_sql(
    f"describe {mrc_table}",
    step_name='describe_mrc_table'
)
display(mrc_describe_df)

required_cols = {
    'ods_commit_ts',
    'ods_insert_ts',
    'ods_deleted_flg',
    'ods_op_csn',
    'ods_op_type',
    'c_nmrc',
    'd_rent',
    'n_amt',
    'commit_date',
}

if len(mrc_describe_df):
    col_name_field = mrc_describe_df.columns[0]
    actual_cols = (
        mrc_describe_df[col_name_field]
        .astype(str)
        .str.strip()
        .str.lower()
    )
    actual_cols = set(c for c in actual_cols if c and not c.startswith('#'))
    missing_cols = sorted(required_cols - actual_cols)
else:
    missing_cols = sorted(required_cols)

if missing_cols:
    print('MISSING COLUMNS:', missing_cols)
else:
    print('Все обязательные колонки найдены.')

In [ ]:
sql_tech_profile = f"""
with base as (
    select
        cast(c_nmrc as string) as c_nmrc,
        cast(d_rent as date) as d_rent_dt,
        cast(n_amt as double) as n_amt_num,
        cast(ods_insert_ts as timestamp) as ods_insert_ts_ts,
        cast(ods_commit_ts as timestamp) as ods_commit_ts_ts,
        cast(commit_date as date) as commit_date_dt,
        cast(ods_op_type as string) as ods_op_type,
        coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
    from {mrc_table}
)
select
    count(*) as total_rows,
    sum(case when c_nmrc is null or trim(c_nmrc) = '' then 1 else 0 end) as c_nmrc_null_cnt,
    sum(case when d_rent_dt is null then 1 else 0 end) as d_rent_null_or_bad_cnt,
    sum(case when n_amt_num is null then 1 else 0 end) as n_amt_null_or_bad_cnt,
    sum(case when ods_deleted_flg in ('1', 'Y', 'y') then 1 else 0 end) as deleted_rows,
    min(d_rent_dt) as min_d_rent,
    max(d_rent_dt) as max_d_rent,
    min(commit_date_dt) as min_commit_date,
    max(commit_date_dt) as max_commit_date,
    min(ods_insert_ts_ts) as min_ods_insert_ts,
    max(ods_insert_ts_ts) as max_ods_insert_ts,
    min(ods_commit_ts_ts) as min_ods_commit_ts,
    max(ods_commit_ts_ts) as max_ods_commit_ts
from base
"""

tech_profile_df = run_sql(sql_tech_profile, step_name='tech_profile')
display(tech_profile_df.T)

sql_op_type = f"""
select
    coalesce(cast(ods_op_type as string), '<NULL>') as ods_op_type,
    count(*) as row_cnt
from {mrc_table}
group by 1
order by row_cnt desc
"""

op_type_df = run_sql(sql_op_type, step_name='op_type_distribution')
display(op_type_df)

In [ ]:
## 2) Профиль качества n_amt

sql_amt_profile = f"""
with base as (
    select
        cast(c_nmrc as string) as c_nmrc,
        cast(d_rent as date) as d_rent_dt,
        cast(n_amt as double) as n_amt_num,
        coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
    from {mrc_table}
), active as (
    select *
    from base
    where ods_deleted_flg not in ('1', 'Y', 'y')
)
select
    count(*) as active_rows,
    sum(case when n_amt_num is null then 1 else 0 end) as amt_null_cnt,
    sum(case when n_amt_num = 0 then 1 else 0 end) as amt_zero_cnt,
    sum(case when n_amt_num < 0 then 1 else 0 end) as amt_negative_cnt,
    min(n_amt_num) as amt_min,
    percentile_approx(n_amt_num, 0.50) as amt_p50,
    percentile_approx(n_amt_num, 0.90) as amt_p90,
    percentile_approx(n_amt_num, 0.95) as amt_p95,
    percentile_approx(n_amt_num, 0.99) as amt_p99,
    max(n_amt_num) as amt_max,
    avg(n_amt_num) as amt_avg,
    stddev_samp(n_amt_num) as amt_std,
    sum(n_amt_num) as amt_total
from active
"""

amt_profile_df = run_sql(sql_amt_profile, step_name='amt_profile')
display(amt_profile_df.T)

sql_amt_top_abs = f"""
select
    cast(c_nmrc as string) as c_nmrc,
    cast(d_rent as date) as d_rent_dt,
    cast(n_amt as double) as n_amt_num,
    cast(ods_commit_ts as timestamp) as ods_commit_ts,
    cast(ods_insert_ts as timestamp) as ods_insert_ts
from {mrc_table}
where cast(n_amt as double) is not null
order by abs(cast(n_amt as double)) desc
limit 50
"""

amt_top_abs_df = run_sql(sql_amt_top_abs, step_name='amt_top_abs_50')
display(amt_top_abs_df)

In [ ]:
## 3) Дубли и конфликтующие суммы по ключу c_nmrc + d_rent

sql_dup_summary = f"""
with base as (
    select
        cast(c_nmrc as string) as c_nmrc,
        cast(d_rent as date) as d_rent_dt,
        cast(n_amt as double) as n_amt_num,
        coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
    from {mrc_table}
    where c_nmrc is not null
      and cast(d_rent as date) is not null
), active as (
    select *
    from base
    where ods_deleted_flg not in ('1', 'Y', 'y')
), grouped as (
    select
        c_nmrc,
        d_rent_dt,
        count(*) as row_cnt,
        count(distinct n_amt_num) as distinct_amt_cnt
    from active
    group by c_nmrc, d_rent_dt
)
select
    (select count(*) from active) as active_rows,
    count(*) as key_cnt,
    sum(case when row_cnt > 1 then 1 else 0 end) as duplicate_key_cnt,
    sum(case when row_cnt > 1 then row_cnt - 1 else 0 end) as duplicate_row_overhead,
    sum(case when distinct_amt_cnt > 1 then 1 else 0 end) as conflict_key_cnt
from grouped
"""

dup_summary_df = run_sql(sql_dup_summary, step_name='dup_summary')
display(dup_summary_df.T)

sql_dup_conflicts_top = f"""
with base as (
    select
        cast(c_nmrc as string) as c_nmrc,
        cast(d_rent as date) as d_rent_dt,
        cast(n_amt as double) as n_amt_num,
        coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg,
        cast(ods_commit_ts as timestamp) as ods_commit_ts,
        cast(ods_insert_ts as timestamp) as ods_insert_ts,
        cast(ods_op_csn as decimal(38, 0)) as ods_op_csn
    from {mrc_table}
    where c_nmrc is not null
      and cast(d_rent as date) is not null
), active as (
    select *
    from base
    where ods_deleted_flg not in ('1', 'Y', 'y')
), conflicts as (
    select c_nmrc, d_rent_dt
    from active
    group by c_nmrc, d_rent_dt
    having count(distinct n_amt_num) > 1
)
select a.*
from active a
join conflicts c
  on c.c_nmrc = a.c_nmrc
 and c.d_rent_dt = a.d_rent_dt
order by a.d_rent_dt desc, a.c_nmrc
limit 200
"""

dup_conflicts_top_df = run_sql(sql_dup_conflicts_top, step_name='dup_conflicts_top')
display(dup_conflicts_top_df)

In [ ]:
## 4) Базовая помесячная агрегация из новой таблицы (после дедупа)

sql_monthly_mrc = f"""
with base as (
    select
        cast(c_nmrc as string) as c_nmrc,
        cast(d_rent as date) as d_rent_dt,
        cast(n_amt as double) as n_amt_num,
        cast(ods_commit_ts as timestamp) as ods_commit_ts,
        cast(ods_insert_ts as timestamp) as ods_insert_ts,
        cast(ods_op_csn as decimal(38, 0)) as ods_op_csn,
        coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
    from {mrc_table}
    where c_nmrc is not null
      and cast(d_rent as date) is not null
), active as (
    select *
    from base
    where ods_deleted_flg not in ('1', 'Y', 'y')
), ranked as (
    select
        *,
        row_number() over (
            partition by c_nmrc, d_rent_dt
            order by coalesce(ods_commit_ts, ods_insert_ts) desc, ods_op_csn desc, ods_insert_ts desc
        ) as rn
    from active
), dedup as (
    select c_nmrc, d_rent_dt, n_amt_num
    from ranked
    where rn = 1
)
select
    trunc(d_rent_dt, 'MM') as month_start,
    count(*) as rows_after_dedup,
    count(distinct c_nmrc) as merchant_cnt,
    sum(n_amt_num) as commission_monthly_new
from dedup
where d_rent_dt between cast('{period_start}' as date) and cast('{period_end}' as date)
group by 1
order by 1
"""

monthly_mrc_df = run_sql(sql_monthly_mrc, step_name='monthly_mrc_after_dedup')
if len(monthly_mrc_df):
    monthly_mrc_df['month_label'] = pd.to_datetime(monthly_mrc_df['month_start']).dt.strftime('%Y-%m')
display(monthly_mrc_df)

In [ ]:
## 5) Маппинг c_nmrc -> inn+agr_id и контроль покрытия

mrc_mapping_cte = f"""
with rent_base as (
    select
        cast(c_nmrc as string) as c_nmrc,
        cast(d_rent as date) as d_rent_dt,
        cast(n_amt as double) as n_amt_num,
        cast(ods_commit_ts as timestamp) as ods_commit_ts,
        cast(ods_insert_ts as timestamp) as ods_insert_ts,
        cast(ods_op_csn as decimal(38, 0)) as ods_op_csn,
        coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
    from {mrc_table}
    where cast(d_rent as date) between cast('{period_start}' as date) and cast('{period_end}' as date)
      and c_nmrc is not null
), rent_ranked as (
    select
        *,
        row_number() over (
            partition by c_nmrc, d_rent_dt
            order by coalesce(ods_commit_ts, ods_insert_ts) desc, ods_op_csn desc, ods_insert_ts desc
        ) as rn
    from rent_base
    where ods_deleted_flg not in ('1', 'Y', 'y')
      and d_rent_dt is not null
), rent_dedup as (
    select c_nmrc, d_rent_dt, n_amt_num
    from rent_ranked
    where rn = 1
), terms_dedup as (
    select distinct
        cast(t.n_agr as string) as n_agr,
        cast(t.c_nmrc as string) as c_nmrc,
        cast(t.d_valid_from as date) as d_valid_from,
        cast(t.d_valid_to as date) as d_valid_to
    from {agr_terms_table} t
    where coalesce(cast(t.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
      and t.c_nmrc is not null
), agreements_dedup as (
    select distinct
        cast(a.n_agr as string) as n_agr,
        cast(a.abs_agr_id as string) as agr_id,
        cast(a.n_cmp_client as string) as n_cmp_client,
        cast(a.d_valid_from as date) as d_valid_from,
        cast(a.d_valid_to as date) as d_valid_to
    from {agreements_table} a
    where coalesce(cast(a.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
), companies_dedup as (
    select distinct
        cast(c.n_cmp as string) as n_cmp,
        regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn
    from {companies_table} c
    where coalesce(cast(c.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
), mapped_raw as (
    select
        r.c_nmrc,
        r.d_rent_dt,
        r.n_amt_num,
        regexp_replace(trim(cast(c.inn as string)), '[^0-9]', '') as inn_key,
        cast(a.agr_id as string) as agr_id_key
    from rent_dedup r
    left join terms_dedup t
      on t.c_nmrc = r.c_nmrc
     and r.d_rent_dt between t.d_valid_from and coalesce(t.d_valid_to, cast('2999-12-31' as date))
    left join agreements_dedup a
      on a.n_agr = t.n_agr
     and r.d_rent_dt between a.d_valid_from and coalesce(a.d_valid_to, cast('2999-12-31' as date))
    left join companies_dedup c
      on c.n_cmp = a.n_cmp_client
)
"""

sql_map_quality = f"""
{mrc_mapping_cte}
, map_stats as (
    select
        c_nmrc,
        d_rent_dt,
        count(distinct case when inn_key is not null and agr_id_key is not null then concat_ws('|', inn_key, agr_id_key) end) as valid_key_cnt
    from mapped_raw
    group by c_nmrc, d_rent_dt
)
select
    count(*) as rent_rows_after_dedup,
    sum(case when valid_key_cnt = 0 then 1 else 0 end) as no_mapping_rows,
    sum(case when valid_key_cnt = 1 then 1 else 0 end) as unique_mapping_rows,
    sum(case when valid_key_cnt > 1 then 1 else 0 end) as ambiguous_mapping_rows
from map_stats
"""

map_quality_df = run_sql(sql_map_quality, step_name='map_quality')
if len(map_quality_df):
    total_rows = float(map_quality_df.loc[0, 'rent_rows_after_dedup'])
    map_quality_df['no_mapping_pct'] = (map_quality_df['no_mapping_rows'] / total_rows * 100.0).round(2)
    map_quality_df['unique_mapping_pct'] = (map_quality_df['unique_mapping_rows'] / total_rows * 100.0).round(2)
    map_quality_df['ambiguous_mapping_pct'] = (map_quality_df['ambiguous_mapping_rows'] / total_rows * 100.0).round(2)
display(map_quality_df)

sql_map_ambiguous_top = f"""
{mrc_mapping_cte}
, map_stats as (
    select
        c_nmrc,
        d_rent_dt,
        count(distinct case when inn_key is not null and agr_id_key is not null then concat_ws('|', inn_key, agr_id_key) end) as valid_key_cnt
    from mapped_raw
    group by c_nmrc, d_rent_dt
)
select *
from map_stats
where valid_key_cnt > 1
order by valid_key_cnt desc, d_rent_dt desc, c_nmrc
limit 100
"""

map_ambiguous_top_df = run_sql(sql_map_ambiguous_top, step_name='map_ambiguous_top')
display(map_ambiguous_top_df)

sql_mrc_key_month = f"""
{mrc_mapping_cte}
, map_stats as (
    select
        c_nmrc,
        d_rent_dt,
        count(distinct case when inn_key is not null and agr_id_key is not null then concat_ws('|', inn_key, agr_id_key) end) as valid_key_cnt
    from mapped_raw
    group by c_nmrc, d_rent_dt
), mapped_unique as (
    select
        mr.c_nmrc,
        mr.d_rent_dt,
        mr.n_amt_num,
        max(mr.inn_key) as inn_key,
        max(mr.agr_id_key) as agr_id_key
    from mapped_raw mr
    join map_stats ms
      on ms.c_nmrc = mr.c_nmrc
     and ms.d_rent_dt = mr.d_rent_dt
    where ms.valid_key_cnt = 1
      and mr.inn_key is not null
      and mr.agr_id_key is not null
    group by mr.c_nmrc, mr.d_rent_dt, mr.n_amt_num
)
select
    trunc(d_rent_dt, 'MM') as month_start,
    inn_key,
    agr_id_key,
    sum(n_amt_num) as commission_monthly_new
from mapped_unique
group by 1, 2, 3
order by 1, 2, 3
"""

mrc_key_month_df = run_sql(sql_mrc_key_month, step_name='mrc_key_month')
if len(mrc_key_month_df):
    mrc_key_month_df['month_label'] = pd.to_datetime(mrc_key_month_df['month_start']).dt.strftime('%Y-%m')
display(mrc_key_month_df.head(30))
print('mrc_key_month rows:', len(mrc_key_month_df))

In [ ]:
## 6) Загрузка Excel и формирование key-month агрегата (inn+agr_id)

excel_col_map = {
    'inn_col': ['ИНН', 'inn', 'c_inn'],
    'agr_col': ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id'],
    'comm_monthly_col': [
        'Комиссия в месяц',
        'Комиссия CN (₽ в месяц)',
        'Комиссия (₽ в месяц)',
        'Комиссия \n(₽ в месяц)',
        'Комиссия (руб в месяц)',
    ],
}


def load_excel_key_month(month_label, excel_path, excel_header):
    ex = pd.read_excel(excel_path, header=excel_header)
    resolved = {k: pick_col_robust(ex.columns, v) for k, v in excel_col_map.items()}
    missing = [k for k, v in resolved.items() if v is None]
    if missing:
        raise ValueError(f'[{month_label}] Не найдены колонки Excel: {missing}. Доступные: {list(ex.columns)}')

    ex['inn_key'] = ex[resolved['inn_col']].apply(normalize_inn_q1)
    ex['agr_id_key'] = ex[resolved['agr_col']].apply(normalize_agr_q1)
    ex['commission_monthly_excel'] = to_num_series(ex[resolved['comm_monthly_col']])
    ex['month_label'] = month_label

    ex_agg = (
        ex.dropna(subset=['inn_key', 'agr_id_key'])
          .groupby(['month_label', 'inn_key', 'agr_id_key'], as_index=False)
          .agg({'commission_monthly_excel': 'max'})
    )
    return ex_agg, resolved


excel_key_month_frames = []
excel_resolved_map = {}

for month_label, excel_path in excel_reference_by_month.items():
    excel_header = int(excel_header_by_month.get(month_label, excel_header_default))
    ex_agg_df, resolved_cols = load_excel_key_month(month_label, excel_path, excel_header)
    excel_key_month_frames.append(ex_agg_df)
    excel_resolved_map[month_label] = resolved_cols
    print(f'[{month_label}] loaded rows={len(ex_agg_df):,}; cols={resolved_cols}')

excel_key_month_df = (
    pd.concat(excel_key_month_frames, ignore_index=True)
    if excel_key_month_frames else pd.DataFrame(columns=['month_label', 'inn_key', 'agr_id_key', 'commission_monthly_excel'])
)

display(excel_key_month_df.head(30))
print('excel_key_month rows:', len(excel_key_month_df))

In [ ]:
## 7) Выбор одного общего мерчанта и помесячная сверка (Excel vs новая таблица)

available_months = sorted(excel_reference_by_month.keys())
months_df = pd.DataFrame({'month_label': available_months})

if manual_inn_key and manual_agr_id_key:
    selected_inn_key = normalize_inn_q1(manual_inn_key)
    selected_agr_id_key = normalize_agr_q1(manual_agr_id_key)
    selected_mode = 'manual'
else:
    keys_intersection = None
    for month_label in available_months:
        month_keys = set(
            excel_key_month_df.loc[
                excel_key_month_df['month_label'] == month_label,
                ['inn_key', 'agr_id_key']
            ].dropna().apply(tuple, axis=1).tolist()
        )
        keys_intersection = month_keys if keys_intersection is None else (keys_intersection & month_keys)

    if not keys_intersection:
        raise RuntimeError('Не найдено ключей inn+agr_id, присутствующих во всех Excel-месяцах.')

    intersection_df = pd.DataFrame(list(keys_intersection), columns=['inn_key', 'agr_id_key'])
    intersection_rank_df = (
        excel_key_month_df
        .merge(intersection_df, on=['inn_key', 'agr_id_key'], how='inner')
        .groupby(['inn_key', 'agr_id_key'], as_index=False)
        .agg(total_commission_excel=('commission_monthly_excel', 'sum'))
        .sort_values('total_commission_excel', ascending=False)
    )

    selected_inn_key = intersection_rank_df.iloc[0]['inn_key']
    selected_agr_id_key = intersection_rank_df.iloc[0]['agr_id_key']
    selected_mode = 'auto_intersection_top'

print('selected_mode:', selected_mode)
print('selected_inn_key:', selected_inn_key)
print('selected_agr_id_key:', selected_agr_id_key)

selected_excel_df = (
    excel_key_month_df[
        (excel_key_month_df['inn_key'] == selected_inn_key)
        & (excel_key_month_df['agr_id_key'] == selected_agr_id_key)
    ][['month_label', 'commission_monthly_excel']]
    .groupby('month_label', as_index=False)
    .agg(commission_monthly_excel=('commission_monthly_excel', 'max'))
)

selected_mrc_df = (
    mrc_key_month_df[
        (mrc_key_month_df['inn_key'] == selected_inn_key)
        & (mrc_key_month_df['agr_id_key'] == selected_agr_id_key)
    ][['month_label', 'commission_monthly_new']]
    .groupby('month_label', as_index=False)
    .agg(commission_monthly_new=('commission_monthly_new', 'sum'))
)

selected_compare_df = (
    months_df
    .merge(selected_excel_df, on='month_label', how='left')
    .merge(selected_mrc_df, on='month_label', how='left')
)
selected_compare_df['commission_monthly_excel'] = selected_compare_df['commission_monthly_excel'].fillna(0.0)
selected_compare_df['commission_monthly_new'] = selected_compare_df['commission_monthly_new'].fillna(0.0)
selected_compare_df['delta_abs'] = selected_compare_df['commission_monthly_new'] - selected_compare_df['commission_monthly_excel']
selected_compare_df['delta_pct'] = selected_compare_df.apply(
    lambda r: safe_divergence_pct(r['delta_abs'], r['commission_monthly_excel']),
    axis=1,
)

selected_compare_df

In [ ]:
## 8) Сверка по месяцу (общий итог + пересечение ключей)

mrc_monthly_total_df = (
    mrc_key_month_df.groupby('month_label', as_index=False)
    .agg(commission_monthly_new=('commission_monthly_new', 'sum'))
)
excel_monthly_total_df = (
    excel_key_month_df.groupby('month_label', as_index=False)
    .agg(commission_monthly_excel=('commission_monthly_excel', 'sum'))
)

monthly_total_compare_df = (
    months_df
    .merge(excel_monthly_total_df, on='month_label', how='left')
    .merge(mrc_monthly_total_df, on='month_label', how='left')
)
monthly_total_compare_df['commission_monthly_excel'] = monthly_total_compare_df['commission_monthly_excel'].fillna(0.0)
monthly_total_compare_df['commission_monthly_new'] = monthly_total_compare_df['commission_monthly_new'].fillna(0.0)
monthly_total_compare_df['delta_abs'] = monthly_total_compare_df['commission_monthly_new'] - monthly_total_compare_df['commission_monthly_excel']
monthly_total_compare_df['delta_pct'] = monthly_total_compare_df.apply(
    lambda r: safe_divergence_pct(r['delta_abs'], r['commission_monthly_excel']),
    axis=1,
)

key_month_inner_df = mrc_key_month_df.merge(
    excel_key_month_df,
    on=['month_label', 'inn_key', 'agr_id_key'],
    how='inner',
)

monthly_intersection_compare_df = (
    key_month_inner_df.groupby('month_label', as_index=False)
    .agg(
        commission_monthly_excel=('commission_monthly_excel', 'sum'),
        commission_monthly_new=('commission_monthly_new', 'sum'),
        key_cnt=('inn_key', 'count'),
    )
)
monthly_intersection_compare_df['delta_abs'] = (
    monthly_intersection_compare_df['commission_monthly_new'] - monthly_intersection_compare_df['commission_monthly_excel']
)
monthly_intersection_compare_df['delta_pct'] = monthly_intersection_compare_df.apply(
    lambda r: safe_divergence_pct(r['delta_abs'], r['commission_monthly_excel']),
    axis=1,
)

key_delta_top_df = key_month_inner_df.copy()
key_delta_top_df['delta_abs'] = key_delta_top_df['commission_monthly_new'] - key_delta_top_df['commission_monthly_excel']
key_delta_top_df['delta_pct'] = key_delta_top_df.apply(
    lambda r: safe_divergence_pct(r['delta_abs'], r['commission_monthly_excel']),
    axis=1,
)
key_delta_top_df = key_delta_top_df.sort_values('delta_abs', key=lambda s: s.abs(), ascending=False).head(200)

print('Общий итог по месяцам:')
display(monthly_total_compare_df)
print('Итог по пересечению ключей (месяц):')
display(monthly_intersection_compare_df)
print('Топ-200 расхождений по ключам inn+agr_id:')
display(key_delta_top_df[['month_label', 'inn_key', 'agr_id_key', 'commission_monthly_excel', 'commission_monthly_new', 'delta_abs', 'delta_pct']])

In [ ]:
## 9) Экспорт результата и краткий DQ-вердикт

output_dir.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(output_report_path, engine='xlsxwriter') as writer:
    mrc_describe_df.to_excel(writer, sheet_name='describe_mrc_table', index=False)
    tech_profile_df.to_excel(writer, sheet_name='tech_profile', index=False)
    op_type_df.to_excel(writer, sheet_name='op_type_distribution', index=False)
    amt_profile_df.to_excel(writer, sheet_name='amt_profile', index=False)
    amt_top_abs_df.to_excel(writer, sheet_name='amt_top_abs_50', index=False)
    dup_summary_df.to_excel(writer, sheet_name='dup_summary', index=False)
    dup_conflicts_top_df.to_excel(writer, sheet_name='dup_conflicts_top', index=False)
    monthly_mrc_df.to_excel(writer, sheet_name='monthly_mrc_after_dedup', index=False)
    map_quality_df.to_excel(writer, sheet_name='map_quality', index=False)
    map_ambiguous_top_df.to_excel(writer, sheet_name='map_ambiguous_top', index=False)
    mrc_key_month_df.to_excel(writer, sheet_name='mrc_key_month', index=False)
    excel_key_month_df.to_excel(writer, sheet_name='excel_key_month', index=False)
    selected_compare_df.to_excel(writer, sheet_name='selected_key_compare', index=False)
    monthly_total_compare_df.to_excel(writer, sheet_name='monthly_total_compare', index=False)
    monthly_intersection_compare_df.to_excel(writer, sheet_name='monthly_intersection_cmp', index=False)
    key_delta_top_df.to_excel(writer, sheet_name='key_delta_top_200', index=False)

selected_compare_df.to_csv(output_key_compare_csv, index=False)

print(f'Excel report saved: {output_report_path}')
print(f'Selected key compare CSV saved: {output_key_compare_csv}')

# Минимальный auto-вердикт
if len(monthly_intersection_compare_df):
    max_pct = monthly_intersection_compare_df['delta_pct'].replace([np.inf, -np.inf], np.nan).dropna()
    max_pct = float(max_pct.max()) if len(max_pct) else np.nan
else:
    max_pct = np.nan

if pd.isna(max_pct):
    dq_status = 'условно OK (недостаточно данных для delta_pct)'
elif max_pct <= 1.0:
    dq_status = 'OK'
elif max_pct <= 5.0:
    dq_status = 'условно OK'
else:
    dq_status = 'не OK'

print('DQ status:', dq_status)
print('Max monthly intersection delta_pct:', max_pct)